# Random Ansatz: clean vs noisy training

This notebook creates a random variational ansatz (inspired by barren-plateau randomized gate choices), trains a hybrid classifier with and without noise on the same split, and compares metrics.

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import ReverseEstimatorGradient

In [ ]:
def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)
print(f"Using random seed: {RANDOM_SEED}")

In [ ]:
def prepare_data():
    banknote = fetch_ucirepo(id=267)
    X = banknote.data.features.copy()
    y = banknote.data.targets.values.flatten()

    X['interaction'] = X['variance'] * X['skewness']
    y_mapped = 2 * y - 1

    return X.values.astype(np.float32), y_mapped.astype(np.float32)

X_full, y_full = prepare_data()
print(f"Dataset shape: X={X_full.shape}, y={y_full.shape}")

In [ ]:
def random_ansatz(num_qubits, depth, seed=42):
    rng = np.random.default_rng(seed)
    qc = QuantumCircuit(num_qubits)

    one_qubit_gates = ['rx', 'ry', 'rz']
    gate_spec = []
    theta = ParameterVector('θ', num_qubits * depth)
    idx = 0

    for layer in range(depth):
        layer_gates = []
        for q in range(num_qubits):
            gate = str(rng.choice(one_qubit_gates))
            layer_gates.append((q, gate))
            if gate == 'rx':
                qc.rx(theta[idx], q)
            elif gate == 'ry':
                qc.ry(theta[idx], q)
            else:
                qc.rz(theta[idx], q)
            idx += 1

        for q in range(num_qubits - 1):
            qc.cz(q, q + 1)

        gate_spec.append(layer_gates)

    return qc, gate_spec

NUM_QUBITS = 5
ANSATZ_DEPTH = 4
ansatz_circuit, ansatz_spec = random_ansatz(NUM_QUBITS, ANSATZ_DEPTH, seed=RANDOM_SEED)

print('Random ansatz gate sequence:')
for layer_id, layer in enumerate(ansatz_spec, start=1):
    print(f'Layer {layer_id}: {layer}')

ansatz_circuit.draw('mpl')

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit, num_qubits=5):
        super().__init__()
        self.num_qubits = num_qubits

        feature_map = self._create_angle_encoding()
        full_circuit = feature_map.compose(ansatz_circuit)

        observable = SparsePauliOp.from_list([('I' * (num_qubits - 1) + 'Z', 1)])
        estimator = StatevectorEstimator()
        gradient = ReverseEstimatorGradient(estimator)

        self.qnn = EstimatorQNN(
            circuit=full_circuit,
            observables=observable,
            input_params=feature_map.parameters,
            weight_params=ansatz_circuit.parameters,
            estimator=estimator,
            gradient=gradient,
            input_gradients=True,
        )
        self.quantum_layer = TorchConnector(self.qnn)

    def _create_angle_encoding(self):
        feature_map = QuantumCircuit(self.num_qubits)
        x = ParameterVector('x', self.num_qubits)
        for i in range(self.num_qubits):
            feature_map.ry(x[i], i)
        return feature_map

    def forward(self, x):
        return self.quantum_layer(x)


class HybridModelNoise(HybridModel):
    def __init__(self, ansatz_circuit, num_qubits=5, epsilon=0.005, n_gates=10, layers_per_step=2):
        super().__init__(ansatz_circuit, num_qubits=num_qubits)
        self.epsilon = epsilon
        self.n_gates = n_gates
        self.layers_per_step = layers_per_step
        self.p_error = 1 - (1 - epsilon) ** (n_gates * layers_per_step)
        self.sigma_noise = 0.2 * self.p_error

    def forward(self, x):
        f_noiseless = self.quantum_layer(x)
        if not self.training:
            return f_noiseless
        attenuation = 1.0 - self.p_error
        noise = torch.randn_like(f_noiseless) * self.sigma_noise
        return (attenuation * f_noiseless) + noise

In [ ]:
TEST_SIZE = 0.2
EPOCHS = 25
BATCH_SIZE = 32
LEARNING_RATE = 0.01

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_full
)

scaler = MinMaxScaler(feature_range=(-np.pi/4, np.pi/4))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train size: {len(X_train_tensor)}, test size: {len(X_test_tensor)}")

In [ ]:
def train_and_evaluate(model_name, model, train_loader, X_test_tensor, y_test_tensor, epochs, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_function = torch.nn.MSELoss()

    best_acc = 0.0
    best_f1 = 0.0
    best_state = None

    history = {'loss': [], 'acc': [], 'f1': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = loss_function(output, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())

        model.eval()
        with torch.no_grad():
            y_pred = (model(X_test_tensor) > 0).float() * 2 - 1

        y_true_np = y_test_tensor.cpu().numpy().flatten()
        y_pred_np = y_pred.cpu().numpy().flatten()
        current_acc = accuracy_score(y_true_np, y_pred_np)
        current_f1 = f1_score(y_true_np, y_pred_np, pos_label=1)

        if (current_acc > best_acc) or (current_acc == best_acc and current_f1 > best_f1):
            best_acc = current_acc
            best_f1 = current_f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

        history['loss'].append(epoch_loss / max(len(train_loader), 1))
        history['acc'].append(current_acc)
        history['f1'].append(current_f1)

        if (epoch + 1) % 5 == 0:
            print(f"{model_name} | Epoch {epoch + 1}/{epochs} | loss={history['loss'][-1]:.4f} | acc={current_acc:.4f} | f1={current_f1:.4f}")

    model.eval()
    with torch.no_grad():
        final_pred = (model(X_test_tensor) > 0).float() * 2 - 1

    final_true = y_test_tensor.cpu().numpy().flatten()
    final_pred_np = final_pred.cpu().numpy().flatten()

    final_metrics = {
        'model': model_name,
        'final_accuracy': float(accuracy_score(final_true, final_pred_np)),
        'final_f1': float(f1_score(final_true, final_pred_np, pos_label=1)),
        'best_accuracy': float(best_acc),
        'best_f1': float(best_f1),
    }

    return final_metrics, history, final_true, final_pred_np, best_state

In [ ]:
clean_model = HybridModel(ansatz_circuit, num_qubits=NUM_QUBITS)
noisy_model = HybridModelNoise(ansatz_circuit, num_qubits=NUM_QUBITS, epsilon=0.005)

clean_metrics, clean_history, y_true_clean, y_pred_clean, clean_best_state = train_and_evaluate(
    'clean', clean_model, train_loader, X_test_tensor, y_test_tensor, EPOCHS, LEARNING_RATE
)

noisy_metrics, noisy_history, y_true_noisy, y_pred_noisy, noisy_best_state = train_and_evaluate(
    'noisy', noisy_model, train_loader, X_test_tensor, y_test_tensor, EPOCHS, LEARNING_RATE
)

metrics_df = pd.DataFrame([clean_metrics, noisy_metrics])
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_clean = confusion_matrix(y_true_clean, y_pred_clean)
cm_noisy = confusion_matrix(y_true_noisy, y_pred_noisy)

ConfusionMatrixDisplay(confusion_matrix=cm_clean, display_labels=[-1, 1]).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Clean model')

ConfusionMatrixDisplay(confusion_matrix=cm_noisy, display_labels=[-1, 1]).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Noisy model')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(clean_history['acc'], label='clean_acc')
plt.plot(noisy_history['acc'], label='noisy_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Test accuracy per epoch')
plt.legend()
plt.show()

In [ ]:
output_dir = Path('tests/shit_model')
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = output_dir / 'random_ansatz_comparison_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

spec_path = output_dir / 'random_ansatz_gate_spec.json'
with open(spec_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'seed': RANDOM_SEED,
            'num_qubits': NUM_QUBITS,
            'ansatz_depth': ANSATZ_DEPTH,
            'gate_spec': ansatz_spec,
        },
        f,
        indent=2,
    )

if clean_best_state is not None:
    torch.save(clean_best_state, output_dir / 'random_ansatz_clean_best.pth')
if noisy_best_state is not None:
    torch.save(noisy_best_state, output_dir / 'random_ansatz_noisy_best.pth')

print(f'Saved metrics: {metrics_path}')
print(f'Saved ansatz spec: {spec_path}')
print(f'Saved checkpoints in: {output_dir}')